# Business Understanding

**Identifiers & Metadata**

application_id: A unique identifier for each individual loan application.

customer_id: A unique identifier for each customer. A single customer may have multiple applications.

application_date: The date on which the loan application was submitted.

data_batch_id: An identifier for the data processing batch this record belongs to.

**Loan Characteristics**

loan_amount_requested: The principal amount of the loan requested by the applicant.

loan_amount_usd: The requested loan amount converted to US Dollars for standardization.

loan_tenure_months: The duration of the loan repayment period in months.

interest_rate_offered: The annual interest rate offered for the loan.

purpose_of_loan: The stated reason for seeking the loan.

loan_type_*: A set of binary columns indicating the specific type of loan product.

**Applicant Financial Profile**

employment_status: The applicant's current employment situation.

monthly_income: The applicant's stated gross monthly income.

yearly_income: The applicant's stated gross annual income.

annual_bonus: The applicant's declared annual bonus amount.

cibil_score: A credit score (e.g., from CIBIL) representing the applicant's creditworthiness and history. Higher scores indicate better credit health.

existing_emis_monthly: The total amount of Equated Monthly Installments (EMIs) the applicant is currently paying for other existing loans.

debt_to_income_ratio: This ratio helps assess an applicant's ability to manage monthly payments.

credit_utilization_ratio: The ratio of the applicant's outstanding credit card debt to their total credit card limit.

**Applicant Demographics & Personal Information**

applicant_age: The age of the applicant in years at the time of application.

gender_*: A set of one-hot encoded binary columns representing the applicant's gender.

property_ownership_status: The applicant's housing situation.

residential_address: The applicant's provided residential address (likely anonymized or generalized).

number_of_dependents: The number of people financially dependent on the applicant.

**Target Variable**

fraud_flag: This is the key target variable for prediction. It's a binary indicator where 1 signifies a fraudulent application and 0 signifies a legitimate application.

# Libraries Used

In [335]:
# basic libraries
import pandas as pd
import numpy as np

# data preparation libraries
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold

# prediction models libraries
from sklearn.metrics import classification_report
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn import linear_model
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier

# Train Dataset

## Data Treatment

In [336]:
df_train = pd.read_csv('train.csv')

# keep only distinct rows, removing duplicated ones
df_train = df_train.drop_duplicates()

# removing unwanted object features
categ_cols = ['purpose_of_loan', 'employment_status', 'property_ownership_status']
df_train[categ_cols] = df_train[categ_cols].astype('category')

# capitalizing category features
for column in categ_cols:
    df_train[column] = df_train[column].str.capitalize()
    df_train[column] = df_train[column].str.strip()

# removing untrustable dummy columns
dummyDrop = [col for col in df_train.columns if 'loan_type' in col]
df_train = df_train.drop(columns=dummyDrop)

## Dropping

In [337]:
df_train = df_train.drop(columns=['monthly_income', # missing values and redundant with 'yearly_income'
                                  'cibil_score', # perfectly normal, likely not real
                                  'loan_amount_requested', # redundant with 'loan_amount_usd'
                                  'application_date', # decided to not be used
                                  'application_id', # irrelevant for prediction
                                  'customer_id', # irrelevant for prediction
                                  'residential_address', # decided to not be used
                                  'Unnamed: 0', # index
                                  'data_batch_id' # irrelevant
                                  ])


# removing untrustable dummy columns
dummyDrop = [col for col in df_train.columns if 'loan_type' in col]
df_train = df_train.drop(columns=dummyDrop)

## Outliers Removal

In [338]:
# removing outliers by the loan tenure months
q1 = df_train['loan_tenure_months'].quantile(0.25)
q3 = df_train['loan_tenure_months'].quantile(0.75)
iqr = q3-q1
upper_bound = q3 + iqr*1.5

# removing outliers
df_train = df_train[df_train['loan_tenure_months']<=upper_bound]

## Handling Missing Values

### Gender: Imputation

In [339]:
## gender imputation through KNN
def gender_code(row):
    if row['gender_Male'] == 1:
        return 1
    elif row['gender_Other'] == 1:
        return 0
    else:
        None #columns as neither male or other will be treated as unkown

df_train['gender_code'] = df_train.apply(gender_code, axis=1)

# defining known and unknown gender
df_known = df_train[df_train['gender_code'].notna()]
df_unknown = df_train[df_train['gender_code'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['gender_code']

# standardizing the known and unknown selected attributes
scaler_input = StandardScaler()
x_train_scaled = scaler_input.fit_transform(x_train)
x_test_scaled = scaler_input.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_train.loc[df_train['gender_code'].isna(), 'gender_code'] = knn.predict(x_test_scaled)

# recreating the gender label, where "0" is man and "1" is woman
gender_drop = ['gender_Male', 'gender_Other']
df_train = df_train.drop(columns=gender_drop)
df_train = df_train.rename(columns={'gender_code': 'gender_male'})

# 0: woman ; 1: man
df_train['gender_male'].value_counts()

gender_male
0.0    21683
1.0    21008
Name: count, dtype: int64

### Number of Dependents: Imputation

In [340]:
## gender imputation through KNN

# defining known and unknown gender
df_known = df_train[df_train['number_of_dependents'].notna()]
df_unknown = df_train[df_train['number_of_dependents'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['number_of_dependents']

# standardizing the known and unknown selected attributes
scaler_input = StandardScaler()
x_train_scaled = scaler_input.fit_transform(x_train)
x_test_scaled = scaler_input.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_train.loc[df_train['number_of_dependents'].isna(), 'number_of_dependents'] = knn.predict(x_test_scaled)

## Dummying

In [341]:
# transforming categorical features into dummy ones, removing one of the categories to avoid colinearility
df_train = pd.get_dummies(df_train, columns=categ_cols, drop_first=True, dtype=int)

## Preparing dataset for the models

In [342]:
# setting target variable
y = df_train['fraud_flag']

# setting feature columns
x = df_train.drop(columns='fraud_flag')

# setting dummy columns
dummy_cols = [col for col in x if any(sub in col for sub in categ_cols) or col == 'gender_male']
not_dummy_cols = [col for col in x if col not in dummy_cols] 

# spliting train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# scaling feature columns but dummy ones
scaler_train = StandardScaler()
x_train_scaled = pd.DataFrame(
    scaler_train.fit_transform(x_train[not_dummy_cols]), # train the model and scales at same time
    columns=not_dummy_cols, 
    index=x_train.index)

x_test_scaled = pd.DataFrame(
    scaler_train.transform(x_test[not_dummy_cols]), # train the model and scales at same time
    columns=not_dummy_cols, 
    index=x_test.index)

# concatenating the dummy cols and the scaled numeric features
x_train = pd.concat([x_train_scaled, x_train[dummy_cols]], axis=1)
x_test = pd.concat([x_test_scaled, x_test[dummy_cols]], axis=1)

# smoted scaled 
smote = SMOTE(sampling_strategy='minority')
x_train_SMOTE, y_smote = smote.fit_resample(x_train, y_train) #smotes train only

# Model Training and Test

## Logistic Regression (Simple)

In [343]:
# creating model
log_reg_model = linear_model.LogisticRegression(random_state=42, class_weight='balanced')

param_grid = [
    {
        'penalty':['l2'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['lbfgs','newton-cg'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['liblinear'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1', 'l2', 'elasticnet'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['saga'], 
        'max_iter': [1000, 3000] # Aumentar max_iter para garantir convergência do SAGA
    },
    {
        'penalty': ['none'], 
        'C': [1.0], # C é ignorado, mas deve estar presente para compatibilidade
        'solver': ['lbfgs'], 
        'max_iter': [1000, 3000]
    }]

# executing GridSearch
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_log_reg_model = grid_search.best_estimator_ #already returns the best model trained

# prediction
y_pred = best_log_reg_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred), '\n')
print("Classification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results = dict()

results['Logistic Regression'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 86 candidates, totalling 430 fits


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
80 fits failed out of a total of 430.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
70 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File 

Confusion Matrix:
col_0          0     1
fraud_flag            
0           3409  3462
1            843   825 

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.50      0.61      6871
           1       0.19      0.49      0.28      1668

    accuracy                           0.50      8539
   macro avg       0.50      0.50      0.45      8539
weighted avg       0.68      0.50      0.55      8539



In [344]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

penalty l1 
 C 0.1 
 solver saga 
 max_iter 1000


## Logistic Regression w/ Stepwise

In [345]:
# setting stepwise
sfs_log_reg = SequentialFeatureSelector(
    best_log_reg_model,
    scoring='f1',
    cv=None
)

# applying the stepwise model
selected_features = sfs_log_reg.fit(x_train, y_train)

# selecting the stepwised features
x_train_stepwised = x_train[selected_features.get_feature_names_out()]
x_test_stepwised = x_test[selected_features.get_feature_names_out()]

# parameters grid
param_grid = [
    {
        'penalty':['l2'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['lbfgs','newton-cg'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['liblinear'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1', 'l2', 'elasticnet'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['saga'], 
        'max_iter': [1000, 3000] # Aumentar max_iter para garantir convergência do SAGA
    },
    {
        'penalty': ['none'], 
        'C': [1.0], # C é ignorado, mas deve estar presente para compatibilidade
        'solver': ['lbfgs'], 
        'max_iter': [1000, 3000]
    }]

# setting GridSearch model
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# applying the GridSearch model
grid_search.fit(x_train_stepwised, y_train)
best_log_reg_model = grid_search.best_estimator_

# prediction
y_pred = best_log_reg_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Logistic Regression w/ Stepwise'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 86 candidates, totalling 430 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           2147  4724
1            528  1140

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.31      0.45      6871
           1       0.19      0.68      0.30      1668

    accuracy                           0.38      8539
   macro avg       0.50      0.50      0.38      8539
weighted avg       0.68      0.38      0.42      8539



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
80 fits failed out of a total of 430.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
70 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File 

In [346]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

penalty l1 
 C 0.1 
 solver saga 
 max_iter 1000


## Logistic Regression w/ Stepwise and SMOTE Resampling

In [347]:
# stepwising after SMOTE
selected_features_SMOTE = sfs_log_reg.fit(x_train_SMOTE, y_smote) #select features after SMOTE
x_stepwised_SMOTE = x_train_SMOTE[selected_features_SMOTE.get_feature_names_out()]

# selecting the stepwise columns for the test sample
x_test_stepwised = x_test[selected_features_SMOTE.get_feature_names_out()]

# parameters Grid
param_grid = [
    {
        'penalty':['l2'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['lbfgs','newton-cg'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['liblinear'], 
        'max_iter': [1000, 3000]
    },
    {
        'penalty': ['l1', 'l2', 'elasticnet'], 
        'C': np.logspace(-3, 3, 7), 
        'solver': ['saga'], 
        'max_iter': [1000, 3000] # Aumentar max_iter para garantir convergência do SAGA
    },
    {
        'penalty': ['none'], 
        'C': [1.0], # C é ignorado, mas deve estar presente para compatibilidade
        'solver': ['lbfgs'], 
        'max_iter': [1000, 3000]
    }]

# executing GridSearch
grid_search = GridSearchCV(
    estimator=log_reg_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# applying the grid search
grid_search.fit(x_stepwised_SMOTE, y_smote)
best_log_reg_model = grid_search.best_estimator_

# the smoted data should be used only for training, not for tests
y_pred = best_log_reg_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Logistic Regression w/ Stepwise and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 86 candidates, totalling 430 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           1925  4946
1            476  1192

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.28      0.42      6871
           1       0.19      0.71      0.31      1668

    accuracy                           0.37      8539
   macro avg       0.50      0.50      0.36      8539
weighted avg       0.68      0.37      0.39      8539



/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
80 fits failed out of a total of 430.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
70 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File 

In [348]:
print(
    'penalty', best_log_reg_model.penalty, '\n',
    'C', best_log_reg_model.C, '\n',
    'solver', best_log_reg_model.solver, '\n',
    'max_iter', best_log_reg_model.max_iter)

penalty l1 
 C 0.01 
 solver saga 
 max_iter 1000


## KNN (Simple)

In [349]:
# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
knn_model = KNeighborsClassifier(weights='distance')
grid_search=GridSearchCV(knn_model, scoring='f1', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_train, y_train)
knn_best_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = knn_best_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

best_model = knn_best_model

# storing results in dict
results[f'KNN ({n_neighbors})'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6864    7
1            738  930

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      6871
           1       0.99      0.56      0.71      1668

    accuracy                           0.91      8539
   macro avg       0.95      0.78      0.83      8539
weighted avg       0.92      0.91      0.90      8539



In [350]:
print('N Neighbors', n_neighbors)

N Neighbors 29


## KNN w/ Stepwise

In [351]:
# defining the stepwise model
sfs_knn = SequentialFeatureSelector(
    knn_model,
    scoring='f1',
    cv=None
)

# applying the stepwise model
selected_features = sfs_knn.fit(x_train, y_train)

# selecting the stepwised features
x_train_stepwised = x_train[selected_features.get_feature_names_out()]
x_test_stepwised = x_test[selected_features.get_feature_names_out()]

# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
grid_search=GridSearchCV(knn_model, scoring='f1', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_train_stepwised, y_train)
knn_best_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = knn_best_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results[f'KNN ({n_neighbors}) w/ Stepwise'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6849   22
1            741  927

Classification Report:
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      6871
           1       0.98      0.56      0.71      1668

    accuracy                           0.91      8539
   macro avg       0.94      0.78      0.83      8539
weighted avg       0.92      0.91      0.90      8539



In [352]:
print('N Neighbors', n_neighbors)

N Neighbors 28


## KNN w/ Stepwise and SMOTE

In [353]:
# stepwising after SMOTE
selected_features_SMOTE = sfs_knn.fit(x_train_SMOTE, y_smote) #select features after SMOTE
x_stepwised_SMOTE = x_train_SMOTE[selected_features_SMOTE.get_feature_names_out()]

# selecting the stepwise columns for the test sample
x_test_stepwised = x_test[x_stepwised_SMOTE.columns]

# training KNN to predict the flags (before it was for the number of dependents)
kf=KFold(n_splits=5,shuffle=True,random_state=42)
parameter={'n_neighbors': np.arange(2, 30, 1)}
grid_search=GridSearchCV(knn_model, scoring='f1', param_grid=parameter, cv=kf, verbose=1)
grid_search.fit(x_stepwised_SMOTE, y_smote)
knn_best_model = grid_search.best_estimator_
n_neighbors = int(grid_search.best_params_['n_neighbors'])

# predicting with KNN
y_pred = knn_best_model.predict(x_test_stepwised)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results[f'KNN ({n_neighbors}) w/ Stepwise and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 28 candidates, totalling 140 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           5220  1651
1            584  1084

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.76      0.82      6871
           1       0.40      0.65      0.49      1668

    accuracy                           0.74      8539
   macro avg       0.65      0.70      0.66      8539
weighted avg       0.80      0.74      0.76      8539



In [354]:
print('N Neighbors', n_neighbors)

N Neighbors 2


## Decision Tree

In [355]:
# loading the decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# training the decision tree model
dtree_model.fit(x_train, y_train)

# predicting
y_pred = dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree'] = classification_report(y_test, y_pred, output_dict=True)

Confusion Matrix:
col_0          0     1
fraud_flag            
0           5742  1129
1            615  1053

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.84      0.87      6871
           1       0.48      0.63      0.55      1668

    accuracy                           0.80      8539
   macro avg       0.69      0.73      0.71      8539
weighted avg       0.82      0.80      0.81      8539



## Decision Tree w/ Grid Search

In [356]:
# setting the initial hyperparameters
param_grid = {
    'max_depth': [30, 35, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# loading the grid search decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=dtree_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    error_score='raise'
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_dtree_model = grid_search.best_estimator_

# prediction
y_pred = best_dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree w/ GridSearch'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           5762  1109
1            614  1054

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.84      0.87      6871
           1       0.49      0.63      0.55      1668

    accuracy                           0.80      8539
   macro avg       0.70      0.74      0.71      8539
weighted avg       0.82      0.80      0.81      8539



In [357]:
print(
    'max_depth', best_dtree_model.max_depth, '\n',
    'min_samples_splt', best_dtree_model.min_samples_split, '\n',
    'min_samples_leaf', best_dtree_model.min_samples_leaf)

max_depth 40 
 min_samples_splt 2 
 min_samples_leaf 1


## Decision Tree w/ Grid Search and SMOTE

In [358]:
# setting the initial hyperparameters
param_grid = {
    'max_depth': [30, 35, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# loading the grid search decision tree model
dtree_model = tree.DecisionTreeClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=dtree_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train_SMOTE, y_smote)
best_dtree_model = grid_search.best_estimator_

# prediction
y_pred = best_dtree_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Decision Tree w/ GridSearch and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Confusion Matrix:
col_0          0     1
fraud_flag            
0           5488  1383
1            579  1089

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.80      0.85      6871
           1       0.44      0.65      0.53      1668

    accuracy                           0.77      8539
   macro avg       0.67      0.73      0.69      8539
weighted avg       0.81      0.77      0.79      8539



In [359]:
print(
    'max_depth', best_dtree_model.max_depth, '\n',
    'min_samples_splt', best_dtree_model.min_samples_split, '\n',
    'min_samples_leaf', best_dtree_model.min_samples_leaf)

max_depth 40 
 min_samples_splt 2 
 min_samples_leaf 1


## Random Forest w/ GridSearch

In [360]:
# setting the initial hyperparameters
param_grid = {
    'n_estimators': [50, 70],
    'max_depth': [15, 20],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# setting the random forest model
rand_for = RandomForestClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=rand_for,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train, y_train)
best_rand_for_model = grid_search.best_estimator_

# prediction
y_pred = best_rand_for_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Random Forest w/ GridSearch'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6871    0
1            921  747

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94      6871
           1       1.00      0.45      0.62      1668

    accuracy                           0.89      8539
   macro avg       0.94      0.72      0.78      8539
weighted avg       0.90      0.89      0.87      8539



In [361]:
print(
    'n_estimators', best_rand_for_model.n_estimators, '\n'
    'max_depth', best_rand_for_model.max_depth, '\n',
    'min_samples_splt', best_rand_for_model.min_samples_split, '\n',
    'min_samples_leaf', best_rand_for_model.min_samples_leaf, '\n',
    'bootstrap', best_rand_for_model.bootstrap
    )

n_estimators 70 
max_depth 20 
 min_samples_splt 2 
 min_samples_leaf 1 
 bootstrap False


## Random Forest w/ GridSearch and SMOTE

In [362]:
# setting the initial hyperparameters
param_grid = {
    'n_estimators': [80, 90],
    'max_depth': [25, 30],
    'min_samples_split': [2, 3],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# setting the random forest model
rand_for = RandomForestClassifier(random_state=42)

# grid search parameters setting
grid_search = GridSearchCV(
    estimator=rand_for,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

# applying the grid search
grid_search.fit(x_train_SMOTE, y_smote)
best_rand_for_model = grid_search.best_estimator_

# prediction
y_pred = best_rand_for_model.predict(x_test)

print("Confusion Matrix:")
print(pd.crosstab(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# storing results in dict
results['Random Forest w/ GridSearch and SMOTE'] = classification_report(y_test, y_pred, output_dict=True)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Confusion Matrix:
col_0          0    1
fraud_flag           
0           6708  163
1            726  942

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      6871
           1       0.85      0.56      0.68      1668

    accuracy                           0.90      8539
   macro avg       0.88      0.77      0.81      8539
weighted avg       0.89      0.90      0.89      8539



In [363]:
print(
    'n_estimators', best_rand_for_model.n_estimators, '\n'
    'max_depth', best_rand_for_model.max_depth, '\n',
    'min_samples_splt', best_rand_for_model.min_samples_split, '\n',
    'min_samples_leaf', best_rand_for_model.min_samples_leaf, '\n',
    'bootstrap', best_rand_for_model.bootstrap
    )

n_estimators 80 
max_depth 30 
 min_samples_splt 2 
 min_samples_leaf 1 
 bootstrap False


## Prediction Results

In [364]:
# all models
models = list(results.keys())

all_rows = []

for model in models:
    rep = results[model]  # classification_report dict

    row = {}
    row['model'] = model

    # classes 0 and 1
    for label in ['0', '1']:
        for metric in ['precision', 'recall', 'f1-score']:
            row[f'{label}_{metric}'] = rep[label][metric]

    # accuracy
    row['accuracy'] = rep['accuracy']

    # macro avg
    for metric in ['precision', 'recall', 'f1-score']:
        row[f'macro_{metric}'] = rep['macro avg'][metric]

    # weighted avg
    for metric in ['precision', 'recall', 'f1-score']:
        row[f'weighted_{metric}'] = rep['weighted avg'][metric]

    all_rows.append(row)

df_results = pd.DataFrame(all_rows)

df_results


,model,0_precision,0_recall,0_f1-score,1_precision,1_recall,1_f1-score,accuracy,macro_precision,macro_recall,macro_f1-score,weighted_precision,weighted_recall,weighted_f1-score
0,Logistic Regression,0.801740,0.496143,0.612964,0.192442,0.494604,0.277078,0.495843,0.497091,0.495374,0.445021,0.682721,0.495843,0.547352
1,Logistic Regression w/ Stepwise,0.802617,0.312473,0.449822,0.194407,0.683453,0.302708,0.384940,0.498512,0.497963,0.376265,0.683810,0.384940,0.421085
2,Logistic Regression w/ Stepwise and SMOTE,0.801749,0.280163,0.415229,0.194200,0.714628,0.305406,0.365031,0.497975,0.497396,0.360317,0.683071,0.365031,0.393776
3,KNN (29),0.902920,0.998981,0.948525,0.992529,0.557554,0.714012,0.912753,0.947725,0.778268,0.831268,0.920424,0.912753,0.902715
4,KNN (28) w/ Stepwise,0.902372,0.996798,0.947237,0.976818,0.555755,0.708445,0.910645,0.939595,0.776277,0.827841,0.916914,0.910645,0.900592
5,KNN (2) w/ Stepwise and SMOTE,0.899380,0.759715,0.823669,0.396344,0.649880,0.492392,0.738260,0.647862,0.704797,0.658030,0.801117,0.738260,0.758957
6,Decision Tree,0.903256,0.835686,0.868158,0.482585,0.631295,0.547013,0.795761,0.692921,0.733491,0.707586,0.821083,0.795761,0.805426
7,Decision Tree w/ GridSearch,0.903701,0.838597,0.869933,0.487286,0.631894,0.550248,0.798220,0.695494,0.735246,0.710090,0.822359,0.798220,0.807486
8,Decision Tree w/ GridSearch and SMOTE,0.904566,0.798719,0.848354,0.440534,0.652878,0.526087,0.770231,0.672550,0.725798,0.687220,0.813922,0.770231,0.785402
9,Random Forest w/ GridSearch,0.881802,1.000000,0.937189,1.000000,0.447842,0.618634,0.892142,0.940901,0.723921,0.777911,0.904891,0.892142,0.874963


# Test

## Data Treatment

### Data Modelling

In [365]:
df_test = pd.read_csv('test.csv')
df_test = df_test.set_index('Unnamed: 0')
df_test = df_test.drop(columns=['data_batch_id']) ##removing first column, that looks just an random id

# df_test = df_test.drop_duplicates() # there's duplicates but if removed, kaggle doesn't accept the submission

# removing unwanted object features
categ_cols = ['purpose_of_loan', 'employment_status', 'property_ownership_status']
df_test[categ_cols] = df_test[categ_cols].astype('category')

# capitalizing category features
for column in categ_cols:
    df_test[column] = df_test[column].str.capitalize()
    df_test[column] = df_test[column].str.strip()

# removing untrustable dummy columns
dummyDrop = [col for col in df_test.columns if 'loan_type' in col]
df_test = df_test.drop(columns=dummyDrop)

### Outliers Removal

In [366]:
'''# removing outliers by the loan tenure months
q1 = df_test['loan_tenure_months'].quantile(0.25)
q3 = df_test['loan_tenure_months'].quantile(0.75)
iqr = q3-q1
upper_bound = q3 + iqr*1.5

# removing outliers
df_test = df_test[df_test['loan_tenure_months']>=upper_bound]'''

"# removing outliers by the loan tenure months\nq1 = df_test['loan_tenure_months'].quantile(0.25)\nq3 = df_test['loan_tenure_months'].quantile(0.75)\niqr = q3-q1\nupper_bound = q3 + iqr*1.5\n\n# removing outliers\ndf_test = df_test[df_test['loan_tenure_months']>=upper_bound]"

### Handling Missing Values

#### Gender: Imputation

In [367]:
## gender imputation through KNN
def gender_code(row):
    if row['gender_Male'] == 1:
        return 1
    elif row['gender_Other'] == 1:
        return 0
    else:
        None #columns as neither male or other will be treated as unkown

df_test['gender_code'] = df_test.apply(gender_code, axis=1)

# defining known and unknown gender
df_known = df_test[df_test['gender_code'].notna()]
df_unknown = df_test[df_test['gender_code'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['gender_code']

# standardizing the known and unknown selected attributes
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_test.loc[df_test['gender_code'].isna(), 'gender_code'] = knn.predict(x_test_scaled)

# recreating the gender label, where "0" is man and "1" is woman
gender_drop = ['gender_Male', 'gender_Other']
df_test = df_test.drop(columns=gender_drop)
df_test = df_test.rename(columns={'gender_code': 'gender_male'})

# 0: woman ; 1: man
df_test['gender_male'].value_counts()

gender_male
0.0    5547
1.0    5453
Name: count, dtype: int64

#### Number of Dependents: Imputation

In [368]:
# defining known and unknown number of dependents
df_known = df_test[df_test['number_of_dependents'].notna()]
df_unknown = df_test[df_test['number_of_dependents'].isna()]

# defining train columns
cols_train = ['debt_to_income_ratio', 'applicant_age', 'yearly_income', 'annual_bonus']

## using the known data to train the model

# defining the train dataset
x_train = df_known[cols_train]
x_test = df_unknown[cols_train]
y_train = df_known['number_of_dependents']

# standardizing the known and unknown selected attributes
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# training the model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(x_train_scaled, y_train)

# replacing the missing values by the KNN prediction
df_test.loc[df_test['number_of_dependents'].isna(), 'number_of_dependents'] = knn.predict(x_test_scaled)

### Dropping

In [369]:
df_test = df_test.drop(columns=['monthly_income', # missing values and redundant with 'yearly_income'
                                  'cibil_score', # perfectly normal, likely not real
                                  'loan_amount_requested', # redundant with 'loan_amount_usd'
                                  'application_date', # decided to not be used
                                  'application_id', # irrelevant for prediction
                                  'customer_id', # irrelevant for prediction
                                  'residential_address', # decided to not be used
                                  ])


# removing untrustable dummy columns
dummyDrop = [col for col in df_test.columns if 'loan_type' in col]
df_train = df_test.drop(columns=dummyDrop)

### Dummying

In [370]:
# transforming categorical features into dummy ones, removing one of the categories to avoid colinearility
df_test = pd.get_dummies(df_test, columns=categ_cols, drop_first=True, dtype=int)

### Scaling

In [371]:
scaler.feature_names_in_

array(['debt_to_income_ratio', 'applicant_age', 'yearly_income',
       'annual_bonus'], dtype=object)

In [372]:
# setting dummy columns
dummy_cols = [col for col in df_test if any(sub in col for sub in categ_cols) or col == 'gender_male']
not_dummy_cols = [col for col in df_test if col not in dummy_cols] 

# scaling feature columns but dummy ones
x_scaled = pd.DataFrame(
    scaler_train.transform(df_test[not_dummy_cols]), # transform based on the train model
    columns=not_dummy_cols, 
    index=df_test.index)

# concatenating the dummy cols in the scaled numeric features
x_final = pd.concat([x_scaled, df_test[dummy_cols]], axis=1)

## Running Best Model

In [373]:
# the smoted data should be used only for training, not for tests
df_test = df_test.copy()
df_test['fraud_flag'] = best_model.predict(x_final)

### Exporting Prediction for Kaggle

In [374]:
df_test.index.name = "ID"
df_test['fraud_flag'].to_csv('submission_f1.csv', index=True)